In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from flowmap.plot import *

def make_s_curve(n_samples=100, noise=0.0, random_state=None):
    """Generate an S curve dataset.

    Parameters
    ----------
    n_samples : int, default=100
        The number of sample points on the S curve.

    noise : float, default=0.0
        The standard deviation of the gaussian noise.

    random_state : int, Generator instance, or None, default=None
        Determines random number generation for dataset creation. Pass an int
        for reproducible output across multiple function calls.

    Returns
    -------
    X : ndarray of shape (n_samples, 3)
        The points.

    t : ndarray of shape (n_samples,)
        The univariate position of the sample according
        to the main dimension of the points in the manifold.
    """
    # Create a Generator instance
    rng = np.random.default_rng(random_state)

    t = 3 * np.pi * (rng.uniform(size=n_samples) - 0.5)
    X = np.empty(shape=(n_samples, 3), dtype=np.float64)
    X[:, 0] = np.sin(t)
    X[:, 1] = 2.0 * rng.uniform(size=n_samples)
    X[:, 2] = np.sign(t) * (np.cos(t) - 1)
    X += noise * rng.standard_normal(size=(n_samples, 3))
    t = np.squeeze(t)
    return X, t


def compute_s_curve_derivative(t):
    """
    Compute the derivative of the S curve with respect to t.

    Parameters
    ----------
    t : ndarray
        The univariate position of the sample according to the main dimension
        of the points in the manifold.

    Returns
    -------
    dX_dt : ndarray of shape (n_samples, 3)
        The derivatives of the S curve with respect to t for each coordinate.
    """
    # Compute derivatives for each component
    dx0_dt = np.pi * np.cos(t)          # Derivative of sin(t) with respect to t
    dx1_dt = np.zeros_like(t)   # Constant component for y (2.0 * uniform), derivative is zero
    dx2_dt = -np.sin(t) * np.sign(t)  # Derivative of z component with respect to t

    dx0_dy = np.zeros_like(t)    # Constant component for y (2.0 * uniform), derivative is zero
    dx1_dy = 2 * np.ones_like(t)     # Constant component for y (2.0 * uniform), derivative is 1
    dx2_dy = np.zeros_like(t)    # Constant component for y (2.0 * uniform), derivative is zero
    
    # Combine into a single array
    dX_dt = np.stack([dx0_dt, dx1_dt, dx2_dt], axis=1)
    dX_dy = np.stack([dx0_dy, dx1_dy, dx2_dy], axis=1)
    
    return dX_dt, dX_dy


def compute_general_derivative(t, dt_ds):
    """
    Compute the derivative of the S curve with respect to a general parameter s,
    where t is a function of s.

    Parameters
    ----------
    t : ndarray
        The values of t for the S curve.

    dt_ds : ndarray
        The derivative of t with respect to s (dt/ds).

    Returns
    -------
    dX_ds : ndarray of shape (n_samples, 3)
        The derivative of the S curve with respect to s for each coordinate.
    """
    # Compute derivatives for each component with respect to t
    dx0_dt = np.cos(t)          # Derivative of sin(t) with respect to t
    dx1_dt = np.zeros_like(t)   # y-coordinate derivative is zero (constant in this case)
    dx2_dt = -np.sin(t) * np.sign(t)  # Derivative of z-component with respect to t

    # Stack derivatives to form dX/dt
    dX_dt = np.stack([dx0_dt, dx1_dt, dx2_dt], axis=1)

    # Apply chain rule to compute dX/ds
    dX_ds = dX_dt * dt_ds[:, np.newaxis]  # Multiply by dt/ds for each coordinate
    dY_ds = None
    return dX_ds, dY_ds

In [ ]:
X, t = make_s_curve(1000, random_state=42)
plot_3d_scatter(X, t, "", dot_size=30, alpha=0.8)

In [ ]:
def normalize_rows(dX_dt):
    # Compute the L2 norm for each row (with keepdims for proper broadcasting)
    row_norms = np.linalg.norm(dX_dt, axis=1, keepdims=True)
    
    # Avoid division by zero: if a row's norm is zero, set it to 1 (so the row stays unchanged)
    row_norms[row_norms == 0] = 1
    
    # Divide each element in the row by the corresponding row norm
    return dX_dt / row_norms

# Compute derivatives with respect to s
dX_dt, dX_dy = compute_s_curve_derivative(t)
dX_ds = normalize_rows(dX_dt)
plot_3d_quiver(X, dX_ds, t, s=30, alpha=0.7, 
                    arrow_size=0.15, normalize=False, 
                    title="", cmap="viridis", show_colorbar=False, show_axes=False)

In [ ]:
# ------------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------------
rng = np.random.default_rng(42)

# ------------------------------------------------------------------
# Noise levels (position std, velocity std)
# ------------------------------------------------------------------
noise_levels = {
    1:  {"pos": 0.25,  "vel": 0.3},
    2:  {"pos": 0.25,  "vel": 0.6},
    3:  {"pos": 0.25,  "vel": 0.9},
    4:  {"pos": 0.25,  "vel": 1.2},
}

# ------------------------------------------------------------------
# Container for all simulations
# ------------------------------------------------------------------
noisy_simulations = {}

for level, stds in noise_levels.items():
    pos_std = stds["pos"]
    vel_std = stds["vel"]

    noisy_points = X + rng.normal(scale=pos_std, size=X.shape)
    noisy_dX_ds  = dX_ds + rng.normal(scale=vel_std, size=dX_ds.shape)

    noisy_simulations[level] = {
        "X_noisy": noisy_points,
        "V_noisy": noisy_dX_ds,
        "position_noise_std": pos_std,
        "velocity_noise_std": vel_std,
    }

In [ ]:
plot_3d_quiver(noisy_simulations[1]["X_noisy"], noisy_simulations[1]["V_noisy"], t, s=30, alpha=0.7, 
                    arrow_size=0.15, normalize=False, 
                    title="", cmap="viridis", show_colorbar=False, show_axes=False)

plot_3d_quiver(noisy_simulations[2]["X_noisy"], noisy_simulations[2]["V_noisy"], t, s=30, alpha=0.7, 
                    arrow_size=0.15, normalize=False, 
                    title="", cmap="viridis", show_colorbar=False, show_axes=False)

plot_3d_quiver(noisy_simulations[3]["X_noisy"], noisy_simulations[3]["V_noisy"], t, s=30, alpha=0.7, 
                    arrow_size=0.15, normalize=False, 
                    title="", cmap="viridis", show_colorbar=False, show_axes=False)

plot_3d_quiver(noisy_simulations[4]["X_noisy"], noisy_simulations[4]["V_noisy"], t, s=30, alpha=0.7, 
                    arrow_size=0.15, normalize=False, 
                    title="", cmap="viridis", show_colorbar=False, show_axes=False)

In [ ]:
from flowmap import VectorFieldEmbedder, Spline

for i in range(1, 5):

    emb = VectorFieldEmbedder(
        noisy_simulations[i]["X_noisy"],
        noisy_simulations[i]["V_noisy"],
        dist_method="phase",
        embed_kwargs={
            "n_neighbors": 30,
            "min_dist": 0.3,
        },
        dof=30,
        method="umap",
    )

    emb.fit_embedding(1)

    plot_2d_quiver(
        emb.X_emb,
        emb.V_emb,
        t,
        scale=3,
        cmap="viridis",
    )